# Results & Findings
## Explainable AI for Medical Imaging: Counterfactual Explanations for Pneumonia Detection

This notebook displays all outputs produced by the pipeline scripts.

**Run order:**
1. `python scripts/01_download_data.py` — Download Kaggle dataset
2. `python scripts/02_preprocess.py` — Explore data & save stats
3. `python scripts/03_train_classifier.py` — Train ResNet-18 classifier
4. `python scripts/04_evaluate.py` — Test set evaluation
5. `python scripts/05_gradcam.py` — Generate Grad-CAM explanations

In [ ]:
import json
from pathlib import Path
from IPython.display import display, Image as IPImage, Markdown

%matplotlib inline

RESULTS = Path('../experiments/results')
GRADCAM = Path('../outputs/gradcam')

---
## 1. Dataset Overview
*From: `scripts/02_preprocess.py`*

In [ ]:
with open(RESULTS / 'data_summary.json') as f:
    summary = json.load(f)

print('Dataset Summary')
print('=' * 45)
for split in ['train', 'val', 'test']:
    total = sum(summary[split].values())
    print(f"  {split:>5}: {summary[split]['NORMAL']:>5} NORMAL | {summary[split]['PNEUMONIA']:>5} PNEUMONIA | {total} total")

stats = summary['image_stats']
print(f"\nImage sizes (sampled {stats['sample_count']} images):")
print(f"  Width:  {stats['width_min']} - {stats['width_max']} (mean {stats['width_mean']})")
print(f"  Height: {stats['height_min']} - {stats['height_max']} (mean {stats['height_mean']})")

In [ ]:
display(Markdown('### Class Distribution'))
display(IPImage(filename=str(RESULTS / 'class_distribution.png'), width=800))

In [ ]:
display(Markdown('### Sample X-Ray Images'))
display(IPImage(filename=str(RESULTS / 'sample_images.png'), width=900))

In [ ]:
display(Markdown('### Pixel Intensity Distributions'))
display(IPImage(filename=str(RESULTS / 'pixel_distributions.png'), width=800))

---
## 2. Classifier Training
*From: `scripts/03_train_classifier.py`*

In [ ]:
with open(RESULTS / 'training_history.json') as f:
    history = json.load(f)

print(f"Trained for {len(history['train_loss'])} epochs")
print(f"Best val loss:     {min(history['val_loss']):.4f} (epoch {history['val_loss'].index(min(history['val_loss'])) + 1})")
print(f"Best val accuracy: {max(history['val_acc']):.4f}")
print(f"Best val F1:       {max(history['val_f1']):.4f}")
print(f"Best val AUC:      {max(history['val_auc']):.4f}")

In [ ]:
display(Markdown('### Training Curves'))
display(IPImage(filename=str(RESULTS / 'training_curves.png'), width=800))

---
## 3. Test Set Evaluation
*From: `scripts/04_evaluate.py`*

In [ ]:
with open(RESULTS / 'test_metrics.json') as f:
    metrics = json.load(f)

print('Test Set Metrics')
print('=' * 35)
for k, v in metrics.items():
    print(f'  {k:>12}: {v:.4f}')

In [ ]:
display(Markdown('### Confusion Matrix'))
display(IPImage(filename=str(RESULTS / 'confusion_matrix.png'), width=500))

In [ ]:
display(Markdown('### ROC Curve'))
display(IPImage(filename=str(RESULTS / 'roc_curve.png'), width=500))

---
## 4. Grad-CAM Explanations
*From: `scripts/05_gradcam.py`*

Each row shows: Original X-Ray | Grad-CAM Heatmap | Overlay

Grad-CAM highlights which regions the classifier focused on. Note the limitations:
- Shows *where* the model looked, but not *what would need to change*
- Can be sensitive to small input perturbations
- A doctor cannot tell from the heatmap alone what makes the image "pneumonia" vs "normal"

In [ ]:
gradcam_images = sorted(GRADCAM.glob('gradcam_*.png'))

if gradcam_images:
    display(Markdown(f'**{len(gradcam_images)} Grad-CAM explanations generated**'))
    for img_path in gradcam_images:
        label = img_path.stem.split('_')[-1].upper()
        display(Markdown(f'#### True label: {label}'))
        display(IPImage(filename=str(img_path), width=800))
else:
    print('No Grad-CAM outputs found. Run: python scripts/05_gradcam.py')

---
## 5. Key Observations

### Dataset
- (Fill in after running 02_preprocess.py)

### Classifier Performance
- (Fill in after running 04_evaluate.py)

### Grad-CAM Limitations
- Heatmaps show attention regions but don't explain *what would change the diagnosis*
- This motivates counterfactual explanations (next phase)

### Next Steps
- Build Diffusion Autoencoder for counterfactual generation
- Compare counterfactuals vs Grad-CAM side-by-side